#### 3.1.2 Проверка и очистка

**Входы**

- Исходные данные

**Выходы**

- Очищенные данные
- Решения о том, что убрать или исправить

**Что делаем**

Проверим одинаковые строки, подозрительные значения и согласованность связанных полей

##### Проверим данные

Идентификаторов заявок нет, поэтому одинаковые значения могут принадлежать разным заявкам. Проверим число повторов в каждой группе и расположение копий в файле\
Отдельно проверим, встречаются ли одинаковые признаки с разным `loan_status`: это уже не полное совпадение записей

In [ ]:
duplicate_extra = data.duplicated()
duplicate_all = data.duplicated(keep=False)
feature_columns = data.columns.drop('loan_status')
conflicting_groups = (
    data.groupby(list(feature_columns), dropna=False)['loan_status']
        .nunique().gt(1)
)

duplicate_summary = pd.DataFrame({
    'Количество': [
        int(duplicate_extra.sum()),
        int(duplicate_all.sum()),
        len(data.loc[duplicate_all].drop_duplicates()),
        int(conflicting_groups.sum()),
    ]
}, index=[
    'Повторные экземпляры сверх первого',
    'Все строки в повторяющихся группах',
    'Группы полностью одинаковых строк',
    'Группы одинаковых признаков с разными loan_status',
])
display(duplicate_summary)

repeated_groups = [
    positions for positions in data.groupby(list(data.columns), dropna=False).indices.values()
    if len(positions) > 1
]
display(pd.Series([len(positions) for positions in repeated_groups])
        .value_counts().sort_index().rename_axis('Записей в группе')
        .to_frame('Число групп'))
pair_gaps = pd.Series([
    positions[1] - positions[0] for positions in repeated_groups if len(positions) == 2
])
display(pd.Series({
    'Соседних пар': int(pair_gaps.eq(1).sum()),
    'Минимальное расстояние между копиями, строк': pair_gaps.min(),
    'Максимальное расстояние между копиями, строк': pair_gaps.max(),
}).to_frame('Значение'))
display(data.loc[duplicate_all].sort_values(list(data.columns)).head(10))


##### Проверим, согласуются ли значения между собой

Например, стаж не должен быть длиннее жизни человека, а доля кредита от дохода должна быть близка к отношению суммы кредита к доходу\
Ниже — рабочие границы проверки. Возраст 100 лет и начало работы в 14 лет — допущения для поиска подозрительных записей, а не правила автора датасета. Проверяем строки, в которых заполнены нужные поля

| Что проверяем | Условие | Почему |
| --- | --- | --- |
| Возраст | Не больше 100 лет | Более высокие значения требуют проверки |
| Стаж | Не больше 86 лет | 100 лет минус предполагаемый возраст начала работы 14 лет |
| Стаж и возраст | Стаж ≤ возраст − 14 | Предполагаем, что человек начал работать не раньше 14 лет |
| Кредитная история и возраст | История ≤ возраст − 16 | Предполагаем начало кредитной истории не раньше 16 лет |
| Доля кредита от дохода | Отличие от суммы / дохода не больше 0,005 | Учитываем округление доли до сотых |
| Возраст, стаж, кредитная история | Целые значения | Проверяем, записаны ли длительности в полных годах |

###### Условия проверки

Запишем перечисленные ограничения как условия нарушения

In [ ]:
ratio_expected = data['loan_amnt'] / data['person_income']
ratio_difference = (data['loan_percent_income'] - ratio_expected).abs()

checks = {
    'Целочисленность возраста': (
        ['person_age'], data['person_age'] % 1 != 0),
    'Целочисленность стажа': (
        ['person_emp_length'], data['person_emp_length'] % 1 != 0),
    'Целочисленность кредитной истории': (
        ['cb_person_cred_hist_length'], data['cb_person_cred_hist_length'] % 1 != 0),
    'Возраст ≤ 100': (
        ['person_age'], data['person_age'] > 100),
    'Стаж ≤ 86': (
        ['person_emp_length'], data['person_emp_length'] > 86),
    'Стаж ≤ возраст − 14': (
        ['person_age', 'person_emp_length'],
        data['person_emp_length'] > data['person_age'] - 14),
    'Кредитная история ≤ возраст − 16': (
        ['person_age', 'cb_person_cred_hist_length'],
        data['cb_person_cred_hist_length'] > data['person_age'] - 16),
    'Доля кредита соответствует сумме / доходу (погрешность ≤ 0,005)': (
        ['loan_percent_income', 'loan_amnt', 'person_income'],
        ratio_difference > 0.005 + 1e-9),
}



###### Число нарушений

Применяем условия только к строкам с нужными значениями и считаем число нарушений каждой проверки

In [ ]:
constraint_results = []
constraint_violations = pd.DataFrame(index=data.index)
for name, (columns, violation) in checks.items():
    available = data[columns].notna().all(axis=1)
    # Согласованность проверяем только при допустимом абсолютном стаже.
    if name == 'Стаж ≤ возраст − 14':
        available &= data['person_emp_length'] <= 86
    if 'person_income' in columns:
        available &= data['person_income'] > 0
    constraint_violations[name] = available & violation
    constraint_results.append({
        'Проверка': name,
        'Проверено': int(available.sum()),
        'Не проверено': int((~available).sum()),
        'Нарушений': int(constraint_violations[name].sum()),
    })

display(pd.DataFrame(constraint_results).set_index('Проверка').rename_axis(None))


##### Очистим найденные ошибки

Все 165 групп совпадающих строк — пары. Это позволяет предположить ошибочное дублирование при подготовке данных, поэтому оставляем по одной записи из каждой пары. Копии не стоят рядом в файле.

Также убираем записи с недопустимым возрастом. Недопустимый стаж заменяем пропуском; способ заполнения пропусков выберем позже\
Дальше работаем с очищенной таблицей `data`

In [ ]:
data = data_raw.drop_duplicates().copy()
invalid_age = data['person_age'] > 100
data = data.loc[~invalid_age].copy()

invalid_employment = data['person_emp_length'] > 86
data.loc[invalid_employment, 'person_emp_length'] = float('nan')


#### Выводы и решения после проверки корректности

In [ ]:
conclusions = update_findings(conclusions, [
    ('f03', 'В person_emp_length (2,75 % записей) и в loan_int_rate (9.56% записей) есть пропуски', 'Способ обработки пропусков выберем после анализа распределений'),
    ('f07', 'Найдено 165 групп ровно по две полностью одинаковые строки: 330 записей, из них 165 повторных (0,51 % выборки).\nВсе повторы парные; соседних копий нет. Считаем это признаком ошибочного дублирования при подготовке данных', 'Оставить по одной записи из каждой пары, исходя из предположения об ошибочном дублировании'),
    ('f08', 'Возраст превышает рабочую границу 100 лет в 5 записях: 144 года в трёх записях, 123 года в двух', 'Убрать эти 5 записей: их мало, а надёжно восстановить возраст нельзя'),
    ('f09', 'В двух записях стаж равен 123 годам и превышает выбранную границу 86 лет', 'Заменить недопустимый стаж пропусками, сохранив записи'),
    ('f10', 'Для записей со стажем ≤ 86 лет выполняется условие «стаж ≤ возраст − 14»', '—'),
    ('f11', 'Выполняется условие «кредитная история ≤ возраст − 16»', '—'),
    ('f12', 'В 429 записях (1,32 % всей выборки) доля кредита отличается от отношения суммы к доходу более чем на 0,005; одним округлением до сотых эти расхождения не объясняются', 'Проверим на этапе корреляционного анализа'),
    ('f13', 'Возраст, стаж и длина кредитной истории везде, где они указаны, записаны целыми числами', '—'),
    ('f26', 'После промежуточной очистки анализируем 32 411 записей', '—'),
], remove=['f05'])
display(findings_view(conclusions))